In [3]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance

client = QdrantClient(host="localhost", port=6333)

/var/folders/9b/87_d8hyd7mq1rhhh_fbqcgj80000gn/T/ipykernel_97034/3268528058.py:4: UserWarning: Qdrant client version 1.17.0 is incompatible with server version 1.15.4. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  client = QdrantClient(host="localhost", port=6333)


In [4]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
os.environ['USE_TF'] = '0'
os.environ['USE_TORCH'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
from sentence_transformers import SentenceTransformer
# Load the embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")


In [10]:
def search_schema(query_text, db_client, embedding_model, collection_name="table", top_k=5):
    """
    Encodes the query text and performs a similarity search in one step.
    """
    print(f"Encoding query: '{query_text}'...")
    query_vector = embedding_model.encode(query_text).tolist()
    
    print(f"Searching collection '{collection_name}'...")
    search_results = db_client.query_points(
        collection_name=collection_name,
        query=query_vector,
        limit=top_k
    )
    return search_results

# Perform the search using the new function
query = "show me tables related to subscriptions and billing"
results = search_schema(query, client, model)
results


Encoding query: 'show me tables related to subscriptions and billing'...
Searching collection 'table'...


QueryResponse(points=[ScoredPoint(id=2, version=0, score=0.7702948, payload={'table': 'subscriptions', 'description': 'Stores detailed records of user-specific subscriptions, including service names, billing terms, costs, and current activity status.', 'primary_keys': ['sub_id', 'billing_cycle', 'is_active'], 'foreign_keys': [{'column': 'user_id', 'references_table': 'users', 'references_column': 'id'}]}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=3, version=0, score=0.676878, payload={'table': 'subscriptions_old', 'description': 'This table stores legacy records of user subscriptions, detailing service names, financial costs, and usage metrics.', 'primary_keys': ['sub_id'], 'foreign_keys': [{'column': 'user_id', 'references_table': 'users', 'references_column': 'id'}]}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=4, version=0, score=0.55493116, payload={'table': 'usage_logs', 'description': 'Stores historical weekly resource consumption records a

In [11]:
# Clean view of retrieved tables
for i, hit in enumerate(results.points, start=1):
    payload = hit.payload or {}
    print(f"{i}. table={payload.get('table')} | score={hit.score:.4f}")
    print(f"   description={payload.get('description')}")
    print()


1. table=subscriptions | score=0.7703
   description=Stores detailed records of user-specific subscriptions, including service names, billing terms, costs, and current activity status.

2. table=subscriptions_old | score=0.6769
   description=This table stores legacy records of user subscriptions, detailing service names, financial costs, and usage metrics.

3. table=usage_logs | score=0.5549
   description=Stores historical weekly resource consumption records associated with specific subscriptions.

4. table=reminders | score=0.5069
   description=Stores a historical record of notifications or reminders sent to users regarding their subscriptions.

5. table=users | score=0.4688
   description=This table stores individual user account details including credentials and demographic information.



In [12]:
import json

def get_top_context_object(search_results, context_file="context.json"):
    """
    Takes the result with the highest score, extracts its name (table),
    and retrieves the matching object from context.json.
    """
    if not search_results or not search_results.points:
        print("No search results found.")
        return None
    
    # 1. Take the highest scoring result (the first one)
    top_hit = search_results.points[0]
    
    # 2. Extract the name
    table_name = top_hit.payload.get("table")
    if not table_name:
        print("Top hit does not contain a table name in its payload.")
        return None
        
    print(f"Top match found: {table_name} (Score: {top_hit.score:.4f})")
    
    # 3. Load context.json
    try:
        with open(context_file, "r") as f:
            context_data = json.load(f)
    except FileNotFoundError:
        print(f"Could not find {context_file}")
        return None
        
    # 4. Return the specific object with the matching key embedded
    obj = context_data.get(table_name)
    if isinstance(obj, dict):
        obj["table_name"] = table_name
    return obj

# Test the function by passing in the results
top_context = get_top_context_object(results)
top_context


Top match found: subscriptions (Score: 0.7703)


{'table_description': 'Stores detailed records of user-specific subscriptions, including service names, billing terms, costs, and current activity status.',
 'columns': [{'column_name': 'sub_id',
   'data_type': 'integer',
   'is_indexed': False,
   'description': 'The unique primary identifier for each subscription record.'},
  {'column_name': 'user_id',
   'data_type': 'integer',
   'is_indexed': False,
   'description': 'The foreign key linking the subscription to a specific user account.'},
  {'column_name': 'sub_name',
   'data_type': 'character varying',
   'is_indexed': False,
   'description': 'The name of the service provider or platform being subscribed to.'},
  {'column_name': 'cost',
   'data_type': 'numeric',
   'is_indexed': False,
   'description': 'The monetary amount charged for the subscription service per billing cycle.'},
  {'column_name': 'category',
   'data_type': 'character varying',
   'is_indexed': False,
   'description': 'The industry or type classification 